# Gold layer v6 — generic, language-aware, context-validated, local Ollama

This Gold layer is generic for Dutch and English documents. It selects compact evidence locally, sends one evidence pack to Ollama, and then validates top-term contexts deterministically.

Key properties:
- No document-specific terms or rules.
- Forces generated summaries and explanations to follow the detected document language.
- Produces 10–15 meaningful top terms when enough evidence exists.
- Replaces weak or mismatched top-term contexts with evidence snippets that actually contain the term.
- Uses Silver NLP as suggestions only.
- Compatible with larger Ollama models later by changing `LOCAL_MODEL`.


In [ ]:

from pathlib import Path
import json
import re
import time
import requests
from datetime import datetime
from collections import Counter, defaultdict

# =========================
# Configuration
# =========================
DOCUMENT_ID = "doc_01"

LOCAL_MODEL = "qwen2.5:3b-instruct"   # Later: "qwen2.5:7b-instruct" or "qwen2.5:14b-instruct" --> irm https://ollama.com/install.ps1 | iex
OLLAMA_URL = "http://localhost:11434/api/generate"

BASE_DIR = Path("../../Data")
SILVER_DIR = BASE_DIR / "silver"
SILVER_NLP_DIR = BASE_DIR / "silver_nlp"
GOLD_DIR = BASE_DIR / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

MAX_EVIDENCE_CHARS = 18000
MAX_SELECTED_CHUNKS = 12
REQUEST_TIMEOUT_SECONDS = 900

TEMPERATURE = 0.1
NUM_CTX = 8192

# Generic NL/EN stopwords and weak terms.
# These are language-general, not specific to any document.
STOPWORDS = {
    # Dutch
    "de","het","een","en","of","in","op","te","van","voor","met","dat","dit","die","als","aan","om","er",
    "is","zijn","wordt","worden","was","waren","heeft","hebben","had","door","bij","uit","naar","ook",
    "maar","dan","dus","kan","kunnen","zal","zullen","moet","moeten","waar","wat","hoe","welke","wie",
    "wanneer","tijdens","alleen","nieuwe","gemaakt","gebruikt","maken","gebruik","bevat","basis",
    "manier","twee","tijd","team","onderzoek","ontwikkelen","ontwikkeld","plaatsen","kaart",
    # English
    "the","a","an","and","or","in","on","to","of","for","with","that","this","these","those","as","by",
    "from","at","it","its","be","is","are","was","were","has","have","had","can","could","should","would",
    "will","may","might","what","how","which","who","when","during","only","new","made","used","use",
    "using","make","contains","basis","way","two","time","team","research","develop","developed"
}

BAD_MARKERS = {
    "figure_caption", "table_start", "table_end", "appendix", "bijlage", "references", "bibliografie"
}

def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=4)

def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def find_file(directory: Path, patterns):
    for p in patterns:
        matches = sorted(directory.glob(p))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No file found in {directory} for patterns: {patterns}")

silver_path = find_file(SILVER_DIR, [f"{DOCUMENT_ID}_silver.json", "*_silver.json"])
silver_nlp_path = find_file(SILVER_NLP_DIR, [f"{DOCUMENT_ID}_silver_nlp.json", "*_silver_nlp.json"])

silver = read_json(silver_path)
silver_nlp = read_json(silver_nlp_path)

detected_language = silver.get("detected_language") or silver_nlp.get("detected_language") or "unknown"

print("Loaded:")
print("Silver:", silver_path)
print("Silver NLP:", silver_nlp_path)
print("Language:", detected_language)


Loaded:
Silver: ..\..\Data\silver\doc_01_silver.json
Silver NLP: ..\..\Data\silver_nlp\doc_01_silver_nlp.json
Language: en


In [54]:

# =========================
# Generic filtering
# =========================

def normalize_term(term: str) -> str:
    term = str(term or "").strip()
    term = re.sub(r"[_\[\]{}<>]+", " ", term)
    term = re.sub(r"\s+", " ", term)
    return term.strip()

def is_bad_keyword(term: str) -> bool:
    if not term:
        return True
    raw = normalize_term(term)
    low = raw.lower()

    if len(low) < 3:
        return True
    if low in STOPWORDS or low in BAD_MARKERS:
        return True
    if any(marker in low for marker in BAD_MARKERS):
        return True
    if re.fullmatch(r"[\d\W_]+", low):
        return True
    if re.fullmatch(r"\d+(\.\d+)*", low):
        return True

    words = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9\-]+", low)
    if not words:
        return True

    # Avoid phrases that are mostly stopwords.
    stop_ratio = sum(1 for w in words if w in STOPWORDS) / max(1, len(words))
    if stop_ratio > 0.55:
        return True

    # Avoid very long broken table fragments.
    if len(words) > 6:
        return True

    # Avoid repeated x/table residue patterns.
    if words.count("x") >= 2:
        return True

    return False

def clean_keyword_suggestions(keyword_suggestions, limit=25):
    cleaned = []
    seen = set()

    for item in keyword_suggestions:
        term = normalize_term(item.get("term", ""))
        key = term.lower()
        if is_bad_keyword(term) or key in seen:
            continue

        # Prefer multi-word version over individual fragments later, but keep useful acronyms.
        seen.add(key)
        cleaned.append({
            "rank": len(cleaned) + 1,
            "term": term,
            "score": item.get("score"),
            "frequency": item.get("frequency"),
            "section_spread": item.get("section_spread"),
            "section_spread_ratio": item.get("section_spread_ratio"),
            "context_example": item.get("context_example", "")
        })

        if len(cleaned) >= limit:
            break

    return cleaned

filtered_keywords = clean_keyword_suggestions(silver_nlp.get("keyword_suggestions", []), limit=30)

print("Filtered keyword suggestions:")
for kw in filtered_keywords[:15]:
    print(f"{kw['rank']}. {kw['term']}")


Filtered keyword suggestions:
1. data
2. model
3. eeg
4. seizure
5. seizures
6. graduation thesis angélique huige
7. graduation thesis angélique
8. thesis angélique huige
9. learning
10. results
11. each
12. graduation thesis
13. thesis angélique
14. angélique huige
15. eeg data


In [55]:

# =========================
# Load chunks / sections
# =========================

def get_chunks_from_silver(silver_obj):
    chunks = silver_obj.get("chunks")
    if isinstance(chunks, list) and chunks:
        return chunks

    chunks_path = SILVER_DIR / f"{DOCUMENT_ID}_chunks.jsonl"
    rows = read_jsonl(chunks_path)
    if rows:
        return rows

    # fallback: create one pseudo chunk from main_text
    main_text = ""
    parts = silver_obj.get("document_parts", {})
    if isinstance(parts, dict):
        main_text = parts.get("main_text", "")
    main_text = main_text or silver_obj.get("main_text", "")

    return [{
        "chunk_id": "chunk_001",
        "source_section_id": "main",
        "source_section_heading": "main_text",
        "text": main_text,
        "word_count": len(main_text.split())
    }]

chunks = get_chunks_from_silver(silver)

def chunk_text(chunk):
    return chunk.get("text") or chunk.get("chunk_text") or chunk.get("content") or ""

def chunk_heading(chunk):
    return chunk.get("source_section_heading") or chunk.get("heading") or ""

def score_chunk(chunk, keywords):
    text = (chunk_heading(chunk) + " " + chunk_text(chunk)).lower()
    score = 0.0

    for idx, kw in enumerate(keywords[:25], start=1):
        term = kw["term"].lower()
        if term and term in text:
            # earlier keyword = stronger
            score += max(1, 28 - idx)

    heading = chunk_heading(chunk).lower()
    # Generic academic/report section signals
    heading_bonus_words = [
        "abstract","summary","samenvatting","introduction","inleiding","method","methode",
        "results","resultaten","discussion","discussie","conclusion","conclusie",
        "requirements","implementation","implementatie","architecture","architectuur",
        "research question","onderzoeksvraag","goal","doelstelling"
    ]
    if any(w in heading for w in heading_bonus_words):
        score += 8

    wc = chunk.get("word_count") or len(chunk_text(chunk).split())
    if 80 <= wc <= 900:
        score += 4
    elif wc < 30:
        score -= 8

    return score

scored_chunks = []
for ch in chunks:
    txt = chunk_text(ch).strip()
    if not txt:
        continue
    scored_chunks.append((score_chunk(ch, filtered_keywords), ch))

selected = [ch for _, ch in sorted(scored_chunks, key=lambda x: x[0], reverse=True)[:MAX_SELECTED_CHUNKS]]

# Put selected chunks back in original order for readability.
chunk_order = {id(ch): i for i, ch in enumerate(chunks)}
selected = sorted(selected, key=lambda ch: chunk_order.get(id(ch), 999999))

print(f"Total chunks: {len(chunks)}")
print(f"Selected chunks: {len(selected)}")
for ch in selected:
    print("-", ch.get("chunk_id"), "|", chunk_heading(ch), "|", ch.get("word_count", len(chunk_text(ch).split())))


Total chunks: 57
Selected chunks: 12
- chunk_005 | 2.3.1 Deep Learning Architectures | 153
- chunk_006 | 2.3.2 Convolutional Neural Networks | 554
- chunk_007 | 2.3.3 Rationale for Model Architecture | 330
- chunk_013 | 2.5.4 Epoching & Extracting Labels | 257
- chunk_027 | 3.2 Business Objectives | 328
- chunk_033 | 3.5.1 Data Collection | 512
- chunk_050 | 4.5.2 Predicted Probabilities | 257
- chunk_051 | 4.5.3 Classification Results | 351
- chunk_052 | 5 Conclusion | 911
- chunk_053 | 5.4 Main question | 410
- chunk_056 | 6.3 Recommendations | 305
- chunk_057 | 6.4 Process Review | 280


In [56]:

# =========================
# Build compact evidence pack
# =========================

def truncate_text(text, max_chars):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."

titlepage_candidates = silver.get("titlepage_candidates") or silver_nlp.get("titlepage_candidates_from_silver") or {}
document_parts = silver.get("document_parts", {}) if isinstance(silver.get("document_parts"), dict) else {}

evidence_parts = []

evidence_parts.append("DOCUMENT INFO")
evidence_parts.append(f"Detected language: {detected_language}")
evidence_parts.append(f"Titlepage candidates from Silver: {json.dumps(titlepage_candidates, ensure_ascii=False)}")

if filtered_keywords:
    evidence_parts.append("\nFILTERED KEYWORD SUGGESTIONS")
    for kw in filtered_keywords[:20]:
        evidence_parts.append(f"- {kw['term']} | freq={kw.get('frequency')} | context={truncate_text(kw.get('context_example',''), 220)}")

evidence_parts.append("\nSELECTED DOCUMENT EVIDENCE")
remaining = MAX_EVIDENCE_CHARS - sum(len(x) for x in evidence_parts)

for ch in selected:
    if remaining <= 1000:
        break
    heading = chunk_heading(ch)
    cid = ch.get("chunk_id", "")
    txt = chunk_text(ch)
    excerpt = truncate_text(txt, min(2200, remaining))
    block = f"\n[{cid}] {heading}\n{excerpt}\n"
    evidence_parts.append(block)
    remaining -= len(block)

evidence_pack = "\n".join(evidence_parts)
print("Evidence pack characters:", len(evidence_pack))
print(evidence_pack[:1500])


Evidence pack characters: 17137
DOCUMENT INFO
Detected language: en
Titlepage candidates from Silver: {"title_candidates": ["Submitted in partial fulfillment of the requirements", "Supervised by:"], "author_candidates": ["A Deep Neural Network for", "the Detection of Epileptic Seizures", "in Comatose Pediatric Patients", "Angélique Huige", "Information and Communication Technology"], "organization_candidates": ["Robert van den Berg, MD, PhD (Erasmus MC)", "Bente Sinke, MSc (HZ UAS)", "Jolène Cijsouw, MSc (HZ UAS)", "Erasmus MC", "Department of Neurology", "HZ University of Applied Sciences"], "document_type_candidates": ["BSc Graduation Thesis", "for the degree of BSc Data Science"], "date_candidates": ["January 14th, 2024"]}

FILTERED KEYWORD SUGGESTIONS
- data | freq=221 | context=scope of this research is set to medically-induced comatose patients in Erasmuc MC's Pediatric Intensive Care Unit. Research is carried out with a selection of data from such patients, consisting of 127 EEG

In [57]:

# =========================
# Ollama call and JSON parsing
# =========================

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        r.raise_for_status()
        return True
    except Exception as e:
        print("Ollama is not reachable. Start it with: ollama serve")
        print("Error:", e)
        return False

def ollama_generate_json(prompt, model=LOCAL_MODEL):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": NUM_CTX
        }
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
    r.raise_for_status()
    return r.json().get("response", "")

def parse_json_safely(text):
    if isinstance(text, dict):
        return text
    text = str(text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    # Fallback: extract first JSON object.
    m = re.search(r"\{.*\}", text, flags=re.S)
    if m:
        return json.loads(m.group(0))
    raise ValueError("Could not parse JSON from model response")

schema_hint = {
    "document_summary": "",
    "top_terms": [
        {"rank": 1, "term": "", "context": "", "evidence": ""}
    ],
    "suggested_entities": {
        "people": [],
        "organizations": [],
        "locations": [],
        "dates": [],
        "methods_tools_or_products": []
    },
    "main_topics": [],
    "results_or_conclusions": [],
    "possible_value_for_knowledge_platform": "",
    "confidence_notes": []
}

output_language_instruction = (
    "Write all natural-language output in Dutch. Keep technical terms in their original form when appropriate."
    if str(detected_language).lower().startswith("nl")
    else "Write all natural-language output in English."
)

prompt = f"""
You are an evidence-based document analysis system.

Task:
Analyze the document evidence and return valid JSON only.

Mandatory language rule:
- Detected language: {detected_language}.
- {output_language_instruction}
- Do not write an English summary for a Dutch document. Use natural, fluent language, not literal translations.

Rules:
- Do not invent names, dates, organizations, results, phone numbers, emails, or claims.
- Use the selected evidence only.
- Silver NLP terms are suggestions only; ignore weak or irrelevant suggestions.
- Top terms must be meaningful domain terms, not generic words.
- Return 10 to 15 top_terms if the evidence supports them.
- For each top term, include a short context sentence in the mandatory output language. The context must directly explain or contain the term.
- Keep the output concise but useful for a knowledge management platform.
- If information is uncertain, mention this in confidence_notes.
- Return exactly one JSON object matching this schema:
{json.dumps(schema_hint, ensure_ascii=False, indent=2)}

DOCUMENT EVIDENCE:
{evidence_pack}
"""

start = time.time()

if not check_ollama():
    raise RuntimeError("Ollama is not running.")

raw_response = ollama_generate_json(prompt)
runtime = round(time.time() - start, 2)

print("Runtime seconds:", runtime)
print(raw_response[:1000])



Runtime seconds: 32.73
{
  "document_summary": "",
  "top_terms": [
    {
      "rank": 1,
      "term": "deep learning",
      "context": "The goal of this research is to develop a deep learning binary classification model capable of detecting anomalies in EEG segments from pediatric comatose patients that are indicative of epileptic seizures.",
      "evidence": "[chunk_005], [chunk_033]"
    },
    {
      "rank": 2,
      "term": "convolutional neural networks (CNN)",
      "context": "According to all three reviews, the most common approach in deep learning model architectures for EEG analysis is by far a convolutional neural network (CNN).",
      "evidence": "[chunk_005]"
    },
    {
      "rank": 3,
      "term": "epileptic seizures",
      "context": "Research shows there are various approaches in applying deep learning for EEG analysis. Insights from literature study can be used to answer the first sub-question: Which deep learning model architecture is most suitable for ide

In [58]:
# =========================
# Validate and write Gold output
# =========================

gold = parse_json_safely(raw_response)

# Normalize required fields.
gold.setdefault("document_summary", "")
gold.setdefault("top_terms", [])
gold.setdefault("suggested_entities", {})
gold.setdefault("main_topics", [])
gold.setdefault("results_or_conclusions", [])
gold.setdefault("possible_value_for_knowledge_platform", "")
gold.setdefault("confidence_notes", [])


def sentence_split(text):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if not text:
        return []
    # Generic sentence-ish splitting that works well enough for NL/EN evidence snippets.
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [p.strip() for p in parts if len(p.strip()) > 20]


def find_context_for_term(term, max_chars=320):
    """Find a generic evidence sentence/snippet that actually contains the term.
    This avoids contexts that drift away from the selected top term.
    """
    term = normalize_term(term)
    if not term:
        return ""
    low_term = term.lower()
    candidates = []

    # Search selected chunks first because they are strongest evidence.
    for ch in selected:
        txt = f"{chunk_heading(ch)}. {chunk_text(ch)}"
        low = txt.lower()
        if low_term in low:
            for sent in sentence_split(txt):
                if low_term in sent.lower():
                    candidates.append(sent)
                    break

    # Fallback: Silver NLP keyword examples.
    for kw in filtered_keywords:
        if normalize_term(kw.get("term", "")).lower() == low_term:
            ex = kw.get("context_example", "")
            if ex:
                candidates.append(ex)

    if not candidates:
        return ""

    # Prefer concise snippets.
    best = sorted(candidates, key=lambda x: abs(len(x) - 220))[0]
    return truncate_text(best, max_chars)


def context_matches_term(term, context):
    term = normalize_term(term).lower()
    context = str(context or "").lower()
    if not term or not context:
        return False
    if term in context:
        return True
    # Acronym and simple plural fallback: generic, language-neutral.
    term_no_s = term[:-1] if term.endswith("s") else term
    if len(term_no_s) >= 4 and term_no_s in context:
        return True
    return False


# Re-rank top terms and remove generic/bad terms if model included any.
clean_terms = []
seen = set()
for item in gold.get("top_terms", []):
    if isinstance(item, str):
        item = {"term": item, "context": "", "evidence": ""}
    term = normalize_term(item.get("term", ""))
    key = term.lower()
    if is_bad_keyword(term) or key in seen:
        continue

    context = str(item.get("context", "") or "").strip()
    if not context_matches_term(term, context):
        replacement = find_context_for_term(term)
        if replacement:
            context = replacement

    seen.add(key)
    clean_terms.append({
        "rank": len(clean_terms) + 1,
        "term": term,
        "context": truncate_text(context, 380),
        "evidence": item.get("evidence", "")
    })

# If the model returned too few useful top terms, use deterministic filtered keywords as fallback.
# This keeps the layer useful on small local models while staying generic.
for kw in filtered_keywords:
    if len(clean_terms) >= 15:
        break
    term = normalize_term(kw.get("term", ""))
    key = term.lower()
    if is_bad_keyword(term) or key in seen:
        continue

    context = kw.get("context_example", "") or find_context_for_term(term)
    seen.add(key)
    clean_terms.append({
        "rank": len(clean_terms) + 1,
        "term": term,
        "context": truncate_text(context, 380),
        "evidence": "Silver NLP keyword suggestion"
    })

gold["top_terms"] = clean_terms[:15]

# Keep all generated descriptive fields in the detected language as much as possible.
# The actual language enforcement is done by the prompt; this post-step only avoids empty fields.
if not str(gold.get("document_summary") or "").strip():
    if filtered_keywords:
        if str(detected_language).lower().startswith("nl"):
            gold["document_summary"] = "Dit document behandelt onder meer " + ", ".join(k["term"] for k in filtered_keywords[:4]) + "."
        else:
            gold["document_summary"] = "This document covers topics such as " + ", ".join(k["term"] for k in filtered_keywords[:4]) + "."

# Clean suggested entities generically: remove internal markers and empty values.
clean_entities = {}
for cat, values in (gold.get("suggested_entities") or {}).items():
    if not isinstance(values, list):
        values = [values]
    out = []
    seen_vals = set()
    for v in values:
        v = normalize_term(v)
        if not v:
            continue
        low = v.lower()
        if any(marker in low for marker in BAD_MARKERS):
            continue
        if low in STOPWORDS:
            continue
        if low in seen_vals:
            continue
        seen_vals.add(low)
        out.append(v)
    clean_entities[cat] = out[:15]
gold["suggested_entities"] = clean_entities

gold["@pipeline"] = {
    "document_id": DOCUMENT_ID,
    "processing_layer": "gold",
    "processing_version": "gold_local_llm_v6_generic_refined_ollama",
    "created_at": datetime.now().isoformat(),
    "runtime_seconds": runtime,
    "local_execution_note": "This layer uses a local Ollama model. It can be upgraded by changing LOCAL_MODEL.",
    "model": LOCAL_MODEL,
    "source_silver_version": silver.get("processing_version"),
    "source_silver_nlp_version": silver_nlp.get("processing_version"),
    "important_note": "Silver NLP inputs were used as suggestions only. Final Gold output is generated from compact document evidence and top-term contexts are validated generically."
}

gold_json_path = GOLD_DIR / f"{DOCUMENT_ID}_gold.json"
write_json(gold, gold_json_path)

# Human-readable txt
lines = []
lines.append(f"# Gold analysis - {DOCUMENT_ID}\n")
lines.append("## Samenvatting / Summary\n")
lines.append(str(gold.get("document_summary") or "-").strip() + "\n")

lines.append("## Top termen / Top terms\n")
for t in gold.get("top_terms", []):
    lines.append(f"{t.get('rank')}. **{t.get('term')}** — {t.get('context','')}")
lines.append("")

lines.append("## Suggestielijst mogelijke entities / Suggested possible entities\n")
entities = gold.get("suggested_entities", {}) or {}
for key, values in entities.items():
    lines.append(f"### {key}")
    if values:
        for v in values[:15]:
            lines.append(f"- {v}")
    else:
        lines.append("-")
    lines.append("")

lines.append("## Hoofdonderwerpen / Main topics\n")
for x in gold.get("main_topics", []) or []:
    lines.append(f"- {x}")
lines.append("")

lines.append("## Resultaten of conclusies / Results or conclusions\n")
for x in gold.get("results_or_conclusions", []) or []:
    lines.append(f"- {x}")
lines.append("")

lines.append("## Mogelijke waarde voor kennisplatform / Possible use for knowledge platform\n")
lines.append(str(gold.get("possible_value_for_knowledge_platform") or "-").strip())
lines.append("")

lines.append("## Confidence notes\n")
for x in gold.get("confidence_notes", []) or []:
    lines.append(f"- {x}")

gold_txt_path = GOLD_DIR / f"{DOCUMENT_ID}_gold_summary.txt"
gold_txt_path.write_text("\n".join(lines), encoding="utf-8")

print("Wrote:")
print(gold_json_path)
print(gold_txt_path)


Wrote:
..\..\Data\gold\doc_01_gold.json
..\..\Data\gold\doc_01_gold_summary.txt
